In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [ ]:
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
# sample first character from the model

g = torch.Generator().manual_seed(2147483647)
p = N[0].float()
p = p / p.sum()

m_ixs = torch.multinomial(p, num_samples=5, replacement=True, generator=g)
ixs = [ix.item() for ix in m_ixs]       # [13, 19, 14, 1, 1]
chars_sample = [itos[ix] for ix in ixs] # ['m', 's', 'n', 'a', 'a']

In [ ]:
# sample words from the model (non-vectorized version)

g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    ix = 0 # we need first special token to start a word
    out = []    
    while True:
        p = N[ix].float()
        p = p / p.sum()
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0: break

    print(''.join(out))


In [ ]:
# sample words from the model (vectorized version)

g = torch.Generator().manual_seed(2147483647)
P = N / N.float().sum(1, keepdim=True)

print(P[0].sum())
print(P.shape)

for i in range(10):
    ix = 0 # we need first special token to start a word
    out = []    
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0: break

    print(''.join(out))

In [ ]:
# evaluate the model
log_likelihood = 0.0
n = 0

# model smoothing, try other values also
P = (N+1).float()
P /= P.float().sum(1, keepdim=True)

for w in ['andrejq']:# words: # try evaluate a probability on any other word/name
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1 
    print(f'{ch1}{ch2}: {prob: .4f} {logprob: .4f}')

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n=}')


In [ ]:
# create the training set of bigrams

xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(xs.dtype, ys.dtype)

In [ ]:
# randomly initialize 27 neurons weights W
# each neuron receives 27 inputs xs

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# forward pass
# input to the network: one-hot encoding
xenc = F.one_hot(xs, num_classes=27).float() 

# predict log-counts
logits = xenc @ W

# Softmax:
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)

print(probs.shape)

In [ ]:
def print_layer_info(i, x, y, itos_x, itos_y, probs_i, p, logp, nll):
    print("-"*30)
    print(f'Bigram example: {i+1}: {itos_x}{itos_y} (indexes: {x}, {y})')
    print(f'Neural net input: {x}')
    print(f'Output probabilities: {probs_i}')
    print(f'Label (next char): {y}')
    print(f'Probability assigned by the Neural Net to the correct char: {p.item()}')
    print(f'Log Likelihood: {logp.item()}')
    print(f'Negative Log Likelihood: {nll.item()}')

    
nlls = torch.zeros(5)
for i in range(5):
    # i-th bigram:
    x = xs[i].item() # input char index
    y = ys[i].item() # label char index
    itos_x = itos[x]
    itos_y = itos[y]
    probs_i = probs[i]
    p = probs[i, y]
    logp = torch.log(p)
    nll = -logp
    nlls[i] = nll
    # print_layer_info(i, x, y, itos_x, itos_y, probs_i, p, logp, nll)

print('='*30)
print(f'Average Negative Log Likelihood, i.e. loss: {nlls.mean().item()}')



In [ ]:
# optimized version of calculations
# Check the value of NLL from previous example:
# Average Negative Log Likelihood, i.e. loss: 3.7693049907684326
loss = -probs[torch.arange(5), ys].log().mean() # output: tensor(3.7693)
print(loss.item()) # Try to run it multiple times with Forward + Backward pass

In [ ]:
# backward pass
W.grad = None # set the gradient to zero
loss.backward()
W.data += -0.1 * W.grad

print(W.shape)
print(W.grad.shape)
print(W.shape == W.grad.shape)

In [ ]:
# Dataset setup + Training loop

# Create dataset
xs, ys = [], []

for w in words: # now we want to use all the words
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

print(f'Number of examples: {num}')

# initialize the neural net
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# Gradient Descent
for k in range(100):

    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean()
    print(f'epoch: {k} | loss: {loss}')

    # backward pass
    W.grad = None
    loss.backward()

    # update
    W.data += -25 * W.grad

In [ ]:
# Sample from the neural net model
for i in range(5):
    out = []
    ix = 0

    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
    
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))